# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

## Part 1: Transform and re-rank
Extend your Week 5 retriever with at least one query transformation and a re-ranking step over the retrieved candidates.

In [1]:
import os
import re
import pathlib
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

os.environ["HF_HOME"] = str(
    (pathlib.Path(".") / ".hf_cache").resolve()
)

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\wyatt\Documents\Gradschool\cosc_650\repos\cosc-650-applied-llm-systems\src\week6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5713.48it/s]


In [2]:
DOCS_DIR = pathlib.Path("../week5/gardening_docs")

def load_pdf(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text()

        if text:
            pages.append(text)

    return "\n\n".join(pages)


documents = []

for file_path in DOCS_DIR.glob("*.pdf"):
    documents.append({
        "source": file_path.name,
        "text": load_pdf(file_path)
    })


print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(f"{doc['source']}: {len(doc['text']):,} characters")

Loaded 8 documents
bush_gardening.pdf: 147,544 characters
fruit_gardening.pdf: 24,727 characters
gardening_beds.pdf: 20,128 characters
gardening_pests.pdf: 10,832 characters
gardening_planters.pdf: 34,943 characters
grass_gardening.pdf: 33,280 characters
tree_planting.pdf: 23,515 characters
vegetable_gardening.pdf: 78,865 characters


In [3]:
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

def chunk_documents(documents, chunker):
    chunks = []

    for doc in documents:
        doc_chunks = chunker(doc["text"])

        for chunk_id, chunk in enumerate(doc_chunks):
            chunks.append({
                "source": doc["source"],
                "chunk_id": chunk_id,
                "text": chunk
            })

    return chunks


configs = {
    "by-paragraph": chunk_documents(
        documents,
        chunk_paragraph
    )
}

for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    texts = [chunk["text"] for chunk in chunks]

    e = embedder.encode(
        texts,
        normalize_embeddings=True
    ).astype("float32")

    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)

    return idx

def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = idx.search(qe, k)

    results = []

    for score, i in zip(scores[0], indices[0]):
        if i >= 0:
            results.append({
                "source": chunks[i]["source"],
                "chunk_id": chunks[i]["chunk_id"],
                "text": chunks[i]["text"],
                "score": float(score)
            })

    return results

by-paragraph     -> 200 chunks


In [4]:
indices = {
    name: build(chunks)
    for name, chunks in configs.items()
}

for name, idx in indices.items():
    print(f"{name:16s} -> {idx.ntotal} vectors")

by-paragraph     -> 200 vectors


In [5]:
test_queries = [
    "How much sunlight do vegetables need?",
    "How should I choose a location for a vegetable garden?",
    "What type of soil is best for planting fruit trees?",
    "How should fruit trees be watered after planting?",
    "What are the benefits of using raised garden beds?",
    "How wide should a raised garden bed be?",
    "How can crop rotation help control garden insects?",
    "How should shrubs be selected for different sunlight conditions?",
    "What should I consider when selecting a tree planting site?",
    "How should a lawn be watered and maintained?"
]

In [6]:
relevant_chunks = {
    "How much sunlight do vegetables need?": {
        "by-paragraph": {("vegetable_gardening.pdf", 1)}
    },

    "How should I choose a location for a vegetable garden?": {
        "by-paragraph": {("vegetable_gardening.pdf", 1), ("vegetable_gardening.pdf", 2)}
    },

    "What type of soil is best for planting fruit trees?": {
        "by-paragraph": {("fruit_gardening.pdf", 0)}
    },

    "How should fruit trees be watered after planting?": {
        "by-paragraph": {("fruit_gardening.pdf", 0),("fruit_gardening.pdf", 1)}
    },

    "What are the benefits of using raised garden beds?": {
        "by-paragraph": {("gardening_beds.pdf", 0),("gardening_beds.pdf", 1),("gardening_planters.pdf", 1)}
    },

    "How wide should a raised garden bed be?": {
        "by-paragraph": {("gardening_beds.pdf", 1)}
    },

    "How can crop rotation help control garden insects?": {
        "by-paragraph": {("gardening_pests.pdf", 0),("gardening_pests.pdf", 2)}
    },

    "How should shrubs be selected for different sunlight conditions?": {
        "by-paragraph": {("bush_gardening.pdf", 20),("bush_gardening.pdf", 27),("bush_gardening.pdf", 64)}
    },

    "What should I consider when selecting a tree planting site?": {
        "by-paragraph": {("tree_planting.pdf", 0),("tree_planting.pdf", 1),("tree_planting.pdf", 2)}
    },

    "How should a lawn be watered and maintained?": {
        "by-paragraph": {("grass_gardening.pdf", 9),
                        ("grass_gardening.pdf", 10),
                        ("grass_gardening.pdf", 11),
                        ("grass_gardening.pdf", 13),
                        ("grass_gardening.pdf", 14),
                        ("grass_gardening.pdf", 15)}
    }
}

In [7]:
def evaluate_query(config_name, query, k=3):
    chunks = configs[config_name]
    idx = indices[config_name]

    results = retrieve(idx, chunks, query, k=k)

    retrieved = {
        (result["source"], result["chunk_id"])
        for result in results
    }

    relevant = relevant_chunks[query][config_name]

    true_positives = retrieved & relevant
    
    precision = len(true_positives) / len(retrieved) if retrieved else 0.0
    recall = len(true_positives) / len(relevant) if relevant else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "retrieved": retrieved,
        "relevant": relevant,
        "true_positives": true_positives
    }

Recalculate baseline metrics

In [8]:
import pandas as pd

evaluation_results = []

for query in test_queries:
    for config_name in configs:
        result = evaluate_query(
            config_name,
            query,
            k=3
        )

        evaluation_results.append({
            "query": query,
            "config": config_name,
            "precision@3": result["precision"],
            "recall@3": result["recall"],
            "hit@3": int(len(result["true_positives"]) > 0),
            "relevant_chunks": len(result["relevant"]),
            "true_positives": len(result["true_positives"])
        })

results_df = pd.DataFrame(evaluation_results)

summary_df = (
    results_df
    .groupby("config")
    .agg({
        "precision@3": "mean",
        "recall@3": "mean",
        "hit@3": "mean"
    })
    .round(3)
)

summary_df

,precision@3,recall@3,hit@3
config,,,
by-paragraph,0.5,0.667,0.9


In [9]:
# Week 5 baseline metrics for by-paragraph chunking
baseline_metrics = {
    "precision@3": 0.5,
    "recall@3": 0.667,
    "hit@3": 0.900
}

In [10]:
def gemini_chat(messages, model="gemini-3.5-flash-lite", **kwargs):
    if not os.environ.get("GEMINI_API_KEY"):
        return None

    from openai import OpenAI

    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        **kwargs
    )

    return response.choices[0].message.content

query_rewrite_cache = {}

def multi_query(query, num_queries=2):
    if query in query_rewrite_cache:
        return query_rewrite_cache[query]

    prompt = f"""
Rewrite the following search query in {num_queries} different ways.

The rewrites should preserve the original meaning while using different
wording or terminology that may improve document retrieval.

Return only the rewritten queries, one per line.

Original query:
{query}
"""

    response = gemini_chat(
        [
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=.7
    )

    if response is None:
        return []

    queries = [
        line.strip()
        for line in response.splitlines()
        if line.strip()
    ][:num_queries]

    query_rewrite_cache[query] = queries

    return queries

def multi_query_retrieve(query, config="by-paragraph", k=3):
    rewritten_queries = multi_query(query)

    queries = [query] + rewritten_queries

    results = []

    for q in queries:
        retrieved = retrieve(
            indices[config],
            configs[config],
            q,
            k=k
        )

        results.extend(retrieved)

    unique_results = {}

    for result in results:
        key = (result["source"], result["chunk_id"])
        unique_results[key] = result

    return list(unique_results.values())

testing the multi_query function.

In [11]:
query = test_queries[0]

print("Original:")
print(query)

print("\nRewrites:")
for rewritten_query in multi_query(query):
    print("-", rewritten_query)

Original:
How much sunlight do vegetables need?

Rewrites:
- ideal sun exposure for vegetable plants
- daily light requirements for growing vegetables


In [13]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank(query, candidates, k=None):
    pairs = [
        [query, candidate["text"]]
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for rank, (candidate, score) in enumerate(ranked, start=1):
        result = candidate.copy()
        result["rerank_score"] = float(score)
        result["rerank_rank"] = rank
        results.append(result)

    if k is not None:
        return results[:k]

    return results

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2970.49it/s]


In [14]:
query = test_queries[0]

candidates = multi_query_retrieve(query)

reranked_results = rerank(
    query,
    candidates,
    k=3
)

for result in reranked_results:
    print(
        result["source"],
        result["chunk_id"],
        result["rerank_score"]
    )

vegetable_gardening.pdf 2 -1.229858636856079
vegetable_gardening.pdf 22 -3.109708309173584
vegetable_gardening.pdf 1 -3.552839756011963


In [17]:
def evaluate_multi_query_rerank(config_name, query, k=3):
    candidates = multi_query_retrieve(
        query,
        config=config_name,
        k=k
    )

    results = rerank(
        query,
        candidates,
        k=k
    )

    retrieved = {
        (result["source"], result["chunk_id"])
        for result in results
    }

    relevant = relevant_chunks[query][config_name]

    true_positives = retrieved & relevant

    precision = (
        len(true_positives) / len(retrieved)
        if retrieved else 0.0
    )

    recall = (
        len(true_positives) / len(relevant)
        if relevant else 0.0
    )

    hit = 1 if true_positives else 0

    return {
        "precision": precision,
        "recall": recall,
        "hit": hit,
        "retrieved": retrieved,
        "relevant": relevant,
        "true_positives": true_positives
    }

In [19]:
# Generate and cache rewrites of query
for query in test_queries:
    multi_query(query)

In [25]:
for query in test_queries:
    rewrites = query_rewrite_cache[query]

    transformation_results.append({
        "original_query": query,
        "rewrite_1": rewrites[0],
        "rewrite_2": rewrites[1]
    })

transformation_df = pd.DataFrame(transformation_results)

transformation_df.style.set_properties(
    subset=["original_query", "rewrite_1", "rewrite_2"],
    **{
        "white-space": "normal",
        "word-wrap": "break-word",
        "max-width": "250px",
        "text-align": "left"
    }
)

,original_query,rewrite_1,rewrite_2
0,How much sunlight do vegetables need?,ideal sun exposure for vegetable plants,daily light requirements for growing vegetables
1,How should I choose a location for a vegetable garden?,best site selection factors for growing a vegetable garden,how to pick the optimal spot for a vegetable plot
2,What type of soil is best for planting fruit trees?,Ideal soil composition and characteristics for fruit tree cultivation,Best growing medium and earth types for fruit orchards
3,How should fruit trees be watered after planting?,Post-planting irrigation guidelines for fruit trees,Best watering practices for newly planted fruit trees
4,What are the benefits of using raised garden beds?,Advantages of planting in raised garden beds,Why use elevated garden beds pros and cons
5,How wide should a raised garden bed be?,Ideal width for a raised garden bed,Raised bed gardening width dimensions
6,How can crop rotation help control garden insects?,Methods for using crop rotation to manage garden pests,Natural pest control through garden crop rotation techniques
7,How should shrubs be selected for different sunlight conditions?,Best bushes for shade vs full sun selection guide,How to choose shrubs based on light exposure
8,What should I consider when selecting a tree planting site?,factors to evaluate before planting a tree,best location requirements for tree planting
9,How should a lawn be watered and maintained?,best practices for lawn watering and upkeep,how to properly irrigate and care for grass


Full list of all baseline rankings vs new rankings with multi-query with reranking

In [27]:
for query in test_queries:

    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    # Baseline retrieval
    baseline_results = retrieve(
        indices["by-paragraph"],
        configs["by-paragraph"],
        query,
        k=3
    )

    print("\nBASELINE:")
    for result in baseline_results:
        print(
            result["source"],
            result["chunk_id"]
        )

    # Multi-query retrieval
    candidates = multi_query_retrieve(
        query,
        config="by-paragraph",
        k=3
    )

    # Rerank candidates and select top 3
    reranked_results = rerank(
        query,
        candidates,
        k=3
    )

    print("\nMULTI-QUERY + RERANK:")
    for result in reranked_results:
        print(
            result["source"],
            result["chunk_id"],
            result["rerank_score"]
        )

    print()

QUERY: How much sunlight do vegetables need?

BASELINE:
vegetable_gardening.pdf 2
vegetable_gardening.pdf 1
vegetable_gardening.pdf 25

MULTI-QUERY + RERANK:
vegetable_gardening.pdf 2 -1.229858636856079
vegetable_gardening.pdf 22 -3.109708309173584
vegetable_gardening.pdf 1 -3.552839756011963

QUERY: How should I choose a location for a vegetable garden?

BASELINE:
vegetable_gardening.pdf 1
gardening_planters.pdf 17
vegetable_gardening.pdf 3

MULTI-QUERY + RERANK:
vegetable_gardening.pdf 3 2.846324920654297
gardening_planters.pdf 17 0.7172541618347168
vegetable_gardening.pdf 1 0.5229645371437073

QUERY: What type of soil is best for planting fruit trees?

BASELINE:
gardening_planters.pdf 1
fruit_gardening.pdf 0
fruit_gardening.pdf 2

MULTI-QUERY + RERANK:
fruit_gardening.pdf 0 3.6813805103302
tree_planting.pdf 2 1.3383910655975342
fruit_gardening.pdf 2 0.3483852446079254

QUERY: How should fruit trees be watered after planting?

BASELINE:
fruit_gardening.pdf 1
fruit_gardening.pdf 0
tre

The table below shows the unique candidate chunks produced by multi-query
retrieval and their subsequent cross-encoder re-ranking results. Each candidate
was scored against the original user query. Candidates were then sorted by
their re-ranking score, with the three highest-ranked chunks selected as the
final retrieval results.

The number of candidates varies by query because duplicate chunks retrieved
by the original query and its transformations were removed before re-ranking.

In [21]:
ranking_results = []

for query in test_queries:
    candidates = multi_query_retrieve(
        query,
        config="by-paragraph",
        k=3
    )

    reranked = rerank(
        query,
        candidates
    )

    for result in reranked:
        ranking_results.append({
            "query": query,
            "source": result["source"],
            "chunk_id": result["chunk_id"],
            "rerank_score": round(result["rerank_score"], 4),
            "rerank_rank": result["rerank_rank"],
            "selected_top_3": result["rerank_rank"] <= 3
        })

ranking_df = pd.DataFrame(ranking_results)

ranking_df

,query,source,chunk_id,rerank_score,rerank_rank,selected_top_3
0,How much sunlight do vegetables need?,vegetable_gardening.pdf,2,-1.2299,1,True
1,How much sunlight do vegetables need?,vegetable_gardening.pdf,22,-3.1097,2,True
2,How much sunlight do vegetables need?,vegetable_gardening.pdf,1,-3.5528,3,True
3,How much sunlight do vegetables need?,vegetable_gardening.pdf,25,-6.9157,4,False
4,How much sunlight do vegetables need?,vegetable_gardening.pdf,11,-7.5659,5,False
5,How much sunlight do vegetables need?,vegetable_gardening.pdf,37,-7.6714,6,False
6,How much sunlight do vegetables need?,vegetable_gardening.pdf,27,-7.7285,7,False
7,How should I choose a location for a vegetable...,vegetable_gardening.pdf,3,2.8463,1,True
8,How should I choose a location for a vegetable...,gardening_planters.pdf,17,0.7173,2,True
9,How should I choose a location for a vegetable...,vegetable_gardening.pdf,1,0.5230,3,True


## Part 2: Measure the effect
Run the same query set as Week 5 and report retrieval quality before and after, with per-query deltas. State which query types benefited and which did not.

In [22]:
comparison_results = []

for query in test_queries:
    baseline = evaluate_query(
        "by-paragraph",
        query,
        k=3
    )

    transformed = evaluate_multi_query_rerank(
        "by-paragraph",
        query,
        k=3
    )

    baseline_hit = 1 if baseline["true_positives"] else 0

    comparison_results.append({
        "query": query,

        "baseline_precision": baseline["precision"],
        "new_precision": transformed["precision"],

        "baseline_recall": baseline["recall"],
        "new_recall": transformed["recall"],

        "baseline_hit": baseline_hit,
        "new_hit": transformed["hit"]
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,query,baseline_precision,new_precision,baseline_recall,new_recall,baseline_hit,new_hit
0,How much sunlight do vegetables need?,0.333333,0.333333,1.000000,1.000000,1,1
1,How should I choose a location for a vegetable...,0.333333,0.333333,0.500000,0.500000,1,1
2,What type of soil is best for planting fruit t...,0.333333,0.333333,1.000000,1.000000,1,1
3,How should fruit trees be watered after planting?,0.666667,0.333333,1.000000,0.500000,1,1
4,What are the benefits of using raised garden b...,0.666667,0.666667,0.666667,0.666667,1,1
5,How wide should a raised garden bed be?,0.000000,0.000000,0.000000,0.000000,0,0
6,How can crop rotation help control garden inse...,0.333333,0.333333,0.500000,0.500000,1,1
7,How should shrubs be selected for different su...,0.666667,0.666667,0.666667,0.666667,1,1
8,What should I consider when selecting a tree p...,1.000000,1.000000,1.000000,1.000000,1,1
9,How should a lawn be watered and maintained?,0.666667,0.666667,0.333333,0.333333,1,1


In [23]:
comparison_df["precision_delta"] = (
    comparison_df["new_precision"]
    - comparison_df["baseline_precision"]
)

comparison_df["recall_delta"] = (
    comparison_df["new_recall"]
    - comparison_df["baseline_recall"]
)

comparison_df["hit_delta"] = (
    comparison_df["new_hit"]
    - comparison_df["baseline_hit"]
)

comparison_df

,query,baseline_precision,new_precision,baseline_recall,new_recall,baseline_hit,new_hit,precision_delta,recall_delta,hit_delta
0,How much sunlight do vegetables need?,0.333333,0.333333,1.000000,1.000000,1,1,0.000000,0.0,0
1,How should I choose a location for a vegetable...,0.333333,0.333333,0.500000,0.500000,1,1,0.000000,0.0,0
2,What type of soil is best for planting fruit t...,0.333333,0.333333,1.000000,1.000000,1,1,0.000000,0.0,0
3,How should fruit trees be watered after planting?,0.666667,0.333333,1.000000,0.500000,1,1,-0.333333,-0.5,0
4,What are the benefits of using raised garden b...,0.666667,0.666667,0.666667,0.666667,1,1,0.000000,0.0,0
5,How wide should a raised garden bed be?,0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0
6,How can crop rotation help control garden inse...,0.333333,0.333333,0.500000,0.500000,1,1,0.000000,0.0,0
7,How should shrubs be selected for different su...,0.666667,0.666667,0.666667,0.666667,1,1,0.000000,0.0,0
8,What should I consider when selecting a tree p...,1.000000,1.000000,1.000000,1.000000,1,1,0.000000,0.0,0
9,How should a lawn be watered and maintained?,0.666667,0.666667,0.333333,0.333333,1,1,0.000000,0.0,0


In [24]:
aggregate_results = pd.DataFrame({
    "metric": [
        "Precision@3",
        "Recall@3",
        "Hit@3"
    ],
    "baseline": [
        comparison_df["baseline_precision"].mean(),
        comparison_df["baseline_recall"].mean(),
        comparison_df["baseline_hit"].mean()
    ],
    "multi_query_rerank": [
        comparison_df["new_precision"].mean(),
        comparison_df["new_recall"].mean(),
        comparison_df["new_hit"].mean()
    ]
})

aggregate_results["delta"] = (
    aggregate_results["multi_query_rerank"]
    - aggregate_results["baseline"]
)

aggregate_results = aggregate_results.round(3)

aggregate_results

,metric,baseline,multi_query_rerank,delta
0,Precision@3,0.500,0.467,-0.033
1,Recall@3,0.667,0.617,-0.050
2,Hit@3,0.900,0.900,0.000


## Part 2 Analysis

The multi-query transformation and re-ranking did not improve performance for any of the query types in the experiment. My queries were primarily direct factual or guidance-based questions, like sunlight requirements, soil requirements. Introducing alternative query wording did not improve precision@3, recall@3 and hit@3. I saw that my "How wide should a raised garden bed be?" remained unsuccessful with both baseline and multi-query and re-ranking. 

One of the queries even regressed in performance. "How should fruit trees be watered after planting?", which is a procedural/how-to query about watering fruit trees, specifically after planting. The baseline retrieved both relevant chunks, while the multi-query with re-ranking displaced one of them from the final top three, which ultimately decreased precision@3 from .667 to .333 and recall@3 from 1.0 to .5. These results show that query transformation is not suited well for these type of straight forward questions. In the next part I will dive further into this and investigate. 

## Part 3: Find one failure and explain it
Show at least one query where the transformation made retrieval worse, and explain why the technique misfired, since a transformation is an assumption about the query and assumptions break. If no query regressed, report that result and explain why the transformation held across your set.

My test query "How should fruit trees be watered after planting?" was the one query where the multi-query transformation with re-ranking reduced the retrieval performance. The baseline retriever's precision@3 was .667 and recall@3 was 1.0, which means that two of the three retrieved chunks were relevant and all relevant chunks were retrieved. In my attempt to improve retrieval with multi-query transformation and coss-encoder reranking, precision@3 decreased to .333 and recall@3 followed downward to 0.5. It's worth noting that hit@3 stayed at 1 because the new approach still found one relevant chunk. 

My multi-query transformation generated 2 alternative queries

| Original Query | Rewrite 1 | Rewrite 2 |
|---|---|---|
| How should fruit trees be watered after planting? | Post-planting irrigation guidelines for fruit trees | Best watering practices for newly planted fruit trees |

These rewrites preserved the general intent of the test query, but broadened teh candiate pool by included new synonyms like irrigation, and paraphrasing like 'newly planted'. This retrieved chunks that were semantically similar to the rewritten queries but not began to drift into adjacent gardening topics. During the reranking, one of the competing candidates was ranked above one of the two relevant chunks from fruit_gardening.pdf, which pushed that relevant chunk outside of the top 3. This failure examples shows how rewriting queries can be a delicate process. Rewriting the query can introduce assumptions about proper terminology to represent the user's initial query, which ultimately leads to less relevant candiates during retrieval. I have a hunch that I could mitigate this side effect by having more than 2 query rewrites in my multi-query algorithm. 